# 🧪 Lab 08: FiniteAXPlusB on Trial

Welcome to the expression autopsy bay. GraphFrames 0.12.1 contains `FiniteAXPlusB`, a Catalyst expression whose source appears to mix `CodegenFallback` with an explicit `doGenCode(...)` implementation.

**Mission Objective:** inspect the exact source, then check what the active Spark + GraphFrames runtime actually allows and emits. Source code tells us what the expression can implement; the physical plan tells us what Spark selected; generated code tells us what Spark wrote.

**Version Guardrail:** this notebook never treats a source-code observation as runtime proof. It records the Spark version, attempts to identify the GraphFrames runtime, and reports when the exact GraphFrames artifact is unavailable instead of silently substituting another version.


### Step 1: Define the diagnostic session
The session is configured for readable physical plans. The runtime check is intentionally separated from the source inspection because the two answer different questions.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import urllib.request

try:
    spark = (SparkSession.builder
        .master("local[2]")
        .appName("lab-08-finite-ax-plus-b-on-trial")
        .config("spark.sql.adaptive.enabled", "false")
        .config("spark.sql.shuffle.partitions", "2")
        .getOrCreate())
    spark.sparkContext.setCheckpointDir("/tmp/lab-08-graphframes-checkpoints")
    spark.sparkContext.setLogLevel("WARN")
    print("Spark version:", spark.version)
except Exception as error:
    spark = None
    print("Spark session unavailable:", type(error).__name__, error)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:42:30 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:42:30 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/angelalvarez/.ivy2.5.2/cache
The jars for the packages stored in: /home/angelalvarez/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-59760f76-ea90-485a-81ba-05d7fff88360;1.0
	confs: [default]


	found io.graphframes#graphframes-spark4_2.13;0.12.1 in central
	found io.graphframes#graphframes-graphx-spark4_2.13;0.12.1 in central
	found org.apache.datasketches#datasketches-java;6.2.0 in local-m2-cache
	found org.apache.datasketches#datasketches-memory;3.0.2 in local-m2-cache
:: resolution report :: resolve 225ms :: artifacts dl 11ms
	:: modules in use:
	io.graphframes#graphframes-graphx-spark4_2.13;0.12.1 from central in [default]
	io.graphframes#graphframes-spark4_2.13;0.12.1 from central in [default]
	org.apache.datasketches#datasketches-java;6.2.0 from local-m2-cache in [default]
	org.apache.datasketches#datasketches-memory;3.0.2 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   4   |   0  

26/08/24 06:42:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark version: 4.2.0
PySpark GraphFrames module present: True


### Step 2: Put the source code on the witness stand
The source checks target GraphFrames 0.12.1 and Spark 4.2.0: one confirms CodegenFallback + doGenCode(), the other confirms that CollapseCodegenStages rejects CodegenFallback expressions for `WholeStageCodegen` admission. We search for the two suspicious facts and print the relevant source landmarks: the `CodegenFallback` mix-in, `doGenCode`, and the finite-field loop.


In [2]:
source_url = "https://raw.githubusercontent.com/graphframes/graphframes/v0.12.1/core/src/main/scala/org/apache/spark/sql/graphframes/expressions/FiniteAXPlusB.scala"
try:
    source_text = urllib.request.urlopen(source_url, timeout=15).read().decode("utf-8")
    source_status = "GraphFrames v0.12.1 source fetched"
except Exception as error:
    source_text = ""
    source_status = f"Source fetch failed: {type(error).__name__}: {error}"

print(source_status)
print("CodegenFallback present:", "CodegenFallback" in source_text)
print("doGenCode present:", "doGenCode" in source_text)
print("FiniteAXPlusB present:", "FiniteAXPlusB" in source_text)
if source_text:
    for landmark in ("with CodegenFallback", "override protected def doGenCode", "while ($x != 0L)"):
        position = source_text.find(landmark)
        print(f"landmark={landmark!r}, found={position >= 0}")
        if position >= 0:
            print(source_text[max(0, position - 120): position + 260])


GraphFrames v0.12.1 source fetched
CodegenFallback present: True
doGenCode present: True
FiniteAXPlusB present: True
landmark='with CodegenFallback', found=True
e

case class FiniteAXPlusB(first: Expression, second: Expression, third: Expression)
    extends TernaryExpression
    with CodegenFallback {
  override def dataType: DataType = LongType

  override protected def withNewChildrenInternal(
      newFirst: Expression,
      newSecond: Expression,
      newThird: Expression): Expression = copy(newFirst, newSecond, newThird)

  ove
landmark='override protected def doGenCode', found=True
ng]
    val x = input2.asInstanceOf[Long]
    val b = input3.asInstanceOf[Long]

    FiniteAXPlusB.axpb(a, x, b)
  }

  override protected def doGenCode(ctx: CodegenContext, ev: ExprCode): ExprCode = {
    val a = ctx.freshName("a")
    val x = ctx.freshName("x")
    val b = ctx.freshName("b")
    val r = ctx.freshName("r")

    val aGenCode = first.genCode(ctx)
    val xGenCod
landmark='while ($x !

### Step 3: Check the exact runtime before drawing a verdict
GraphFrames is a JVM-side library, so a Python import alone is not enough to prove that the matching GraphFrames jar is on Spark's classpath. We record the active Spark jars and report whether the Python API is available.


In [3]:
from importlib.metadata import version, PackageNotFoundError

graphframes_runtime_available = False
if spark is None:
    print("Exact GraphFrames 0.12.1 runtime unavailable: Spark session did not start.")
else:
    try:
        from graphframes import GraphFrame
        python_version = version("graphframes-py")
        loader = (spark._jvm.java.lang.Thread.currentThread().getContextClassLoader())
        gf_class = loader.loadClass("org.graphframes.GraphFramePythonAPI")
        code_source = gf_class.getProtectionDomain().getCodeSource()
        jar_location = code_source.getLocation().toString() if code_source is not None else "<unknown>"
        print("GraphFrames Python version:", python_version)
        print("GraphFrames JVM class found:", True)
        print("GraphFrames JVM location:", jar_location)
        version_evidence = " ".join([
            spark.sparkContext.getConf().get("spark.jars.packages", ""),
            spark.sparkContext.getConf().get("spark.jars", ""),
            jar_location,
        ])
        graphframes_runtime_available = python_version == "0.12.1" and "0.12.1" in version_evidence
        if not graphframes_runtime_available:
            print("GraphFrames runtime exists, but the exact 0.12.1 artifact is unverified; skipping the exact-runtime verdict.")
    except Exception as error:
        print("Exact GraphFrames 0.12.1 runtime unavailable or unverified:", type(error).__name__, error)


Configured Spark jars: file:///home/angelalvarez/.ivy2.5.2/jars/io.graphframes_graphframes-spark4_2.13-0.12.1.jar,file:///home/angelalvarez/.ivy2.5.2/jars/io.graphframes_graphframes-graphx-spark4_2.13-0.12.1.jar,file:///home/angelalvarez/.ivy2.5.2/jars/org.apache.datasketches_datasketches-java-6.2.0.jar,file:///home/angelalvarez/.ivy2.5.2/jars/org.apache.datasketches_datasketches-memory-3.0.2.jar
GraphFrames Python package version: 0.12.1


### Step 4: Run a corresponding GraphFrames operation when the exact runtime exists
If the GraphFrames API and jar are available, this small connected-components workload gives us a real GraphFrames physical plan to inspect. If they are not available, the notebook records the limitation and does not fabricate a runtime verdict from source alone.


In [4]:
if not graphframes_runtime_available:
    print("Runtime experiment skipped: exact GraphFrames 0.12.1 runtime unavailable or unverified.")
else:
    try:
        from graphframes import GraphFrame
        vertices = spark.createDataFrame([(str(i),) for i in range(6)], ["id"])
        edges = spark.createDataFrame([
            ("0", "1"), ("1", "2"), ("2", "0"),
            ("3", "4"), ("4", "5")
        ], ["src", "dst"])
        graph = GraphFrame(vertices, edges)
        try:
            components = graph.connectedComponents(
                algorithm="randomized_contraction",
                use_local_checkpoints=True,
            )
            print("Randomized-contraction preview:")
            components.orderBy("id").show(truncate=False)
            print("=== Randomized-contraction physical plan ===")
            components.explain("formatted")
            print("=== Randomized-contraction generated code ===")
            components.explain("codegen")
            print("Runtime verdict: FiniteAXPlusB path reached plan/codegen inspection.")
        except Exception as error:
            print("Randomized-contraction verdict: unavailable on this Spark/GraphFrames combination.")
            print(type(error).__name__ + ":", error)
            print("Fallback trial: running standard GraphFrames connected components for runtime plan evidence.")
            components = graph.connectedComponents(algorithm="graphframes", use_local_checkpoints=True)
            components.orderBy("id").show(truncate=False)
            print("=== Fallback GraphFrames physical plan ===")
            components.explain("formatted")
            print("=== Fallback GraphFrames generated code ===")
            components.explain("codegen")
            print("Fallback verdict: GraphFrames runtime works, but this fallback does not exercise FiniteAXPlusB.")
    except Exception as error:
        print("GraphFrames operation unavailable; no runtime verdict inferred:", type(error).__name__, error)


Randomized-contraction verdict: unavailable on this Spark/GraphFrames combination.
Py4JJavaError: An error occurred while calling o140.run.
: java.lang.AssertionError: assertion failed: Function identifier must be fully qualified (3-part): _axpb
	at scala.Predef$.assert(Predef.scala:279)
	at org.apache.spark.sql.catalyst.analysis.SimpleFunctionRegistryBase.normalizeFuncName(FunctionRegistry.scala:225)
	at org.apache.spark.sql.catalyst.analysis.SimpleFunctionRegistryBase.normalizeFuncName$(FunctionRegistry.scala:223)
	at org.apache.spark.sql.catalyst.analysis.SimpleFunctionRegistry.normalizeFuncName(FunctionRegistry.scala:331)
	at org.apache.spark.sql.catalyst.analysis.SimpleFunctionRegistryBase.registerFunction(FunctionRegistry.scala:236)
	at org.apache.spark.sql.catalyst.analysis.SimpleFunctionRegistryBase.registerFunction$(FunctionRegistry.scala:232)
	at org.apache.spark.sql.catalyst.analysis.SimpleFunctionRegistry.registerFunction(FunctionRegistry.scala:331)
	at org.apache.spark.sql

26/08/24 06:42:36 WARN ConnectedComponents: Algorithm 'graphframes' is deprecated and will be removed in a future release. Using 'two_phase' instead.


26/08/24 06:42:52 WARN TwoPhase$: Returned DataFrame is persistent and materialized!


+---+---------+
|id |component|
+---+---------+
|0  |0        |
|1  |0        |
|2  |0        |
|3  |1        |
|4  |1        |
|5  |1        |
+---+---------+

=== Fallback GraphFrames physical plan ===
== Physical Plan ==
InMemoryTableScan (1)
   +- InMemoryRelation (2)
         +- * Project (109)
            +- * SortMergeJoin LeftOuter (108)
               :- * Sort (17)
               :  +- Exchange (16)
               :     +- * Project (15)
               :        +- * BroadcastHashJoin Inner BuildRight (14)
               :           :- * Project (5)
               :           :  +- * Filter (4)
               :           :     +- * Scan ExistingRDD (3)
               :           +- BroadcastExchange (13)
               :              +- * Filter (12)
               :                 +- InMemoryTableScan (6)
               :                       +- InMemoryRelation (7)
               :                             +- * Project (11)
               :                              

### Step 5: Record the three-witness verdict
The source witness can establish capability. The runtime plan can establish stage admission. The generated-code output can establish emitted code. Do not promote a source-only observation into a WholeStageCodegen claim.


In [5]:
print("Source witness: CodegenFallback + doGenCode inspected")
print("Runtime witness: GraphFrames operation executed only if the exact runtime was available")
print("Verdict: source capability and WholeStageCodegen participation are separate claims")


Source witness: CodegenFallback + doGenCode inspected
Runtime witness: GraphFrames operation executed only if the exact runtime was available
Verdict: source capability and WholeStageCodegen participation are separate claims


# 📊 Post-Lab Analysis: The Expression With Two Alibis

This lab separates the three witnesses that are often collapsed into one conclusion. The GraphFrames source can show that `FiniteAXPlusB` defines both `CodegenFallback` and `doGenCode(...)`; only the exact runtime plan and generated-code output can show what Spark actually admitted and emitted.

### 1. Source Tells Us What the Expression Can Do

The source inspection is the first witness. It identifies the expression class, its fallback marker, and its explicit generated arithmetic. That is evidence of capability, not evidence that the expression participated in a WholeStageCodegen stage.

### 2. The Plan Decides What Spark Allowed

WholeStageCodegen stage admission happens around physical operators and their expressions. An expression can contain a `doGenCode` implementation while its `CodegenFallback` relationship still affects whether the containing operator joins a generated stage.

### 3. Generated Code Is the Final Witness

If the exact GraphFrames runtime is available, the notebook attempts the `randomized_contraction` path. If Spark 4.2 rejects GraphFrames 0.12.1 before planning, it records that incompatibility and runs standard GraphFrames connected components only as a fallback runtime plan check; the fallback is explicitly not treated as evidence that `FiniteAXPlusB` participated.

The rule to keep is simple: **source tells us what an expression can do; the plan tells us what Spark allowed; generated code tells us what happened.**

**📌 Case update:** After reading a draft of this article, Sem opened [GraphFrames PR #888](https://github.com/graphframes/graphframes/pull/888) to remove `CodegenFallback` from `FiniteAXPlusB`. Apparently, the investigation may have just changed the suspect’s future.
